# Lesson 10: Containerized SLM Serving — High-Throughput vLLM, PagedAttention & ONNX Runtime GenAI

In this companion notebook, you will:
1. Simulate **PagedAttention** block allocation vs. contiguous KV-cache allocation.
2. Export a compact open-source transformer (`Qwen/Qwen2.5-0.5B-Instruct` or `gpt2` demo) to ONNX and run inference via **ONNX Runtime**.
3. Inspect Prometheus-style latency and throughput telemetry.

In [ ]:
!pip install -q -U onnx onnxruntime transformers torch

## 1. Export & Run ONNX Runtime Inference on Colab
We export a lightweight sequence model to ONNX and verify identical logits between PyTorch and `onnxruntime.InferenceSession`.

In [ ]:
import time
import numpy as np
import torch
import onnxruntime as ort
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)
model.eval()

inputs = tokenizer("Edge AI runs locally because", return_tensors="pt")
onnx_path = "smollm2_135m.onnx"

torch.onnx.export(
    model,
    (inputs["input_ids"], inputs["attention_mask"]),
    onnx_path,
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={"input_ids": {0: "batch", 1: "seq"}, "attention_mask": {0: "batch", 1: "seq"}, "logits": {0: "batch", 1: "seq"}},
    opset_version=14,
)

sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
t0 = time.perf_counter()
ort_out = sess.run(None, {
    "input_ids": inputs["input_ids"].numpy(),
    "attention_mask": inputs["attention_mask"].numpy(),
})
elapsed_ms = (time.perf_counter() - t0) * 1000.0
next_token_id = int(np.argmax(ort_out[0][0, -1, :]))
print(f"ONNX Runtime inference: {elapsed_ms:.2f} ms | Next token: {tokenizer.decode([next_token_id])!r}")